In [1]:
import torch

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

mps


In [2]:
import os

from data.augment import AudioAugmenter, AugmentConfig
from data.load_data import (
    TwitDataset,
    collate_fn,
    get_class_imbalance,
    make_augmenting_collate,
)
from torch.utils.data import DataLoader

ds = TwitDataset()  # resamples once to an on-disk cache; later runs just read the cache

# Train/test split
train_ds, test_ds = torch.utils.data.random_split(ds, [0.9, 0.1])

augment_config = AugmentConfig(sample_rate=16_000)
train_collate = make_augmenting_collate(AudioAugmenter(augment_config))

NUM_WORKERS = min(4, os.cpu_count() or 1)
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
    pin_memory=(device.type == "cuda"),
)
train_dataloader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=train_collate,
    **loader_kwargs,
)
test_dataloader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    **loader_kwargs,
)

pos_weight = get_class_imbalance(train_ds).to(device)

In [3]:
from models import CNNHead, GaborFilter, GaborNet, SpecAugment

N_FILTERS = 40
SAMPLE_RATE = 16000     # must match the (resampled) audio fed to the model
KERNEL_SIZE = 401       # ~25 ms @ 16 kHz: long enough to resolve ~1 kHz carriers
STRIDE = 160

feat_extract = GaborFilter(n_filters=N_FILTERS, kernel_size=KERNEL_SIZE, sample_rate=SAMPLE_RATE, stride=STRIDE)
head = CNNHead(channels=(16, 32, 64))   # 3-block 2D CNN over the [n_filters, T] map
model = GaborNet(feat_extract, head).to(device)
model.spec_augment = SpecAugment(
    freq_masks=2,
    time_masks=2,
    max_freq_fraction=0.20,
    max_time_fraction=0.15,
).to(device)

In [4]:
from train import Trainer

trainer = Trainer(model, train_dataloader, test_dataloader, pos_weight, device, lr=1e-2, lr_scheduler_kwargs={"eta_min": 0.0})
trainer.train_with_slimming(
    sparsify_epochs=15,
    finetune_epochs=15,
    pruning_ratio=0.32,
    reg=1e-5,
)

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

...

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

In [5]:
trainer.save_model()

PosixPath('checkpoints/cnn/0831-140730.pt')